In [1]:
import sys, json
sys.path.append("../src")
from cdss_rules import apply_rules
from fhir_mapper import build_fhir_bundle
from datetime import date

# Recreate the sample patient used in the smoke tests
sample = {
    "patient_id": "P00042",
    "age": 67, "sex": "M", "race_ethnicity": "White",
    "smoking_status": "Current", "alcohol_use": "Heavy",
    "hpv_status": "Positive",
    "ml_risk_probability": 0.55,
    "nlp_max_urgency": 10, "note_risk_category": "High",
    "nlp_extracted_terms": ["leukoplakia", "non-healing ulcer", "urgent referral"],
    "has_recent_completed_referral": False,
    "last_screening_date": "2023-09-01",
    "last_referral_date": "2026-02-01",
    "has_pathology": False,
    "n_appts": 5, "no_show_rate": 0.40,
}

flags = apply_rules(sample, today=date(2026, 5, 1))
for f in flags:
    print(f"[{f.severity}] {f.label}\n  {f.rationale}\n")

[High] High-Priority Oral Oncology Referral
  Patient has suspicious clinical findings AND established risk factors AND no recent completed specialist referral.

[Medium] High Predicted Oral Cancer Risk (ML)
  ML model predicted oral cancer probability of 0.55.

[High] Biopsy Follow-Up Gap
  Referral was placed >30 days ago but no pathology record exists. Patient may have been lost to follow-up.

[Medium] Oral Cancer Screening Overdue
  High-risk patient has no documented oral cancer screening in the past 12 months.

[Low] High No-Show Risk
  Patient missed 40% of 5 appointments. Outreach may need extra effort.



In [2]:
bundle = build_fhir_bundle(sample, flags)
print(json.dumps(bundle, indent=2, default=str)[:3000])  # truncate for readability

{
  "resourceType": "Bundle",
  "id": "56c5508f-58a3-4808-a673-7616fbd658da",
  "type": "collection",
  "timestamp": "2026-05-15T19:52:00Z",
  "entry": [
    {
      "resource": {
        "resourceType": "Patient",
        "id": "P00042",
        "identifier": [
          {
            "system": "https://oral-onc.local/patients",
            "value": "P00042"
          }
        ],
        "gender": "male",
        "extension": [
          {
            "url": "http://hl7.org/fhir/us/core/StructureDefinition/us-core-race",
            "valueString": "White"
          }
        ],
        "_age": 67
      },
      "fullUrl": "urn:uuid:P00042"
    },
    {
      "resource": {
        "resourceType": "Observation",
        "id": "e1404533-2d71-45df-8e91-342ffcbe4371",
        "status": "final",
        "category": [
          {
            "coding": [
              {
                "system": "http://terminology.hl7.org/CodeSystem/observation-category",
                "code": "social-his

In [3]:
import pandas as pd
from collections import Counter
res_types = [e["resource"]["resourceType"] for e in bundle["entry"]]
pd.Series(Counter(res_types)).rename("count").to_frame()

,count
Patient,1
Observation,3
Condition,3
RiskAssessment,1


In [4]:
from pathlib import Path
out = Path("../docs/example_fhir_bundle.json")
out.parent.mkdir(exist_ok=True)
out.write_text(json.dumps(bundle, indent=2, default=str))
print(f"Saved: {out}")

Saved: ../docs/example_fhir_bundle.json
